知识蒸馏的loss function
- $q(x)$: from student models，$p(x)$: from teacher models
- $\mathcal{L}_{student}公式$
$$
\begin{aligned}
\mathcal{L}_{student} &= \alpha L_{CE} + (1 - \alpha)L_{KD} \\
&= \alpha L_{CE} + (1 - \alpha)T^2D_{KL}
\end{aligned}
$$

## 自定义trainer和training argument

In [1]:
%%capture
!pip install evaluate

In [2]:
from transformers import Trainer, TrainingArguments
import torch
from torch import nn
import torch.nn.functional as F
import numpy as np 

2025-06-14 02:22:33.275112: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749867753.299024     264 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749867753.306027     264 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
class DistillTrainingArguments(TrainingArguments):
    # TrainingArguments: @dataclass
    # 增加两个 KD 所需的参数参数
    def __init__(self, *args, alpha=0.5, temperature=2., **kwargs):
        super().__init__(*args,**kwargs)
        self.alpha = alpha
        self.temperature = temperature

In [4]:
class DistillTrainer(Trainer):
    def __init__(self, *args, teacher_model=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # If you are not using num_items_in_batch when computing your loss, 
        # make sure to overwrite self.model_accepts_loss_kwargs to False.
        self.model_accepts_loss_kwargs = False
        
        # loss (torch.FloatTensor of shape (1,), optional, returned when labels is provided) 
        s_output = model(**inputs)
        s_ce = s_output.loss
        s_logits = s_output.logits

        with torch.no_grad():
            t_output = self.teacher_model(**inputs)
            t_logits = t_output.logits

        loss_kl_func = nn.KLDivLoss(reduction='batchmean')
        loss_kd = self.args.temperature**2 * loss_kl_func(F.log_softmax(s_logits/self.args.temperature, dim=-1), 
                                                     F.softmax(t_logits/self.args.temperature, dim=-1))
        loss = self.args.alpha*s_ce + (1-self.args.alpha)*loss_kd
        return (loss, s_output) if return_outputs else loss

关于transformer model输出相关内容，详细请看：https://huggingface.co/docs/transformers/main_classes/output

## dataset

In [5]:
from datasets import load_dataset
clinc = load_dataset("clinc_oos", "plus")
clinc

DatasetDict({
    train: Dataset({
        features: ['text', 'intent'],
        num_rows: 15250
    })
    validation: Dataset({
        features: ['text', 'intent'],
        num_rows: 3100
    })
    test: Dataset({
        features: ['text', 'intent'],
        num_rows: 5500
    })
})

In [6]:
intents = clinc['train'].features['intent']
num_labels = intents.num_classes
num_labels

151

In [7]:
from transformers import AutoConfig, AutoTokenizer
from transformers import AutoModelForSequenceClassification

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

s_ckpt = 'distilbert-base-uncased'
s_tokenizer = AutoTokenizer.from_pretrained(s_ckpt)

t_ckpt = 'transformersbook/bert-base-uncased-finetuned-clinc'
t_model = AutoModelForSequenceClassification.from_pretrained(t_ckpt).to(device)

In [9]:
def tokenize_function(example):
    return s_tokenizer(example['text'], truncation=True)

clinc_enc = clinc.map(tokenize_function, batched=True)
clinc_enc = clinc_enc.remove_columns(['text'])
clinc_enc = clinc_enc.rename_column('intent', 'labels')
clinc_enc

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 15250
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 3100
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 5500
    })
})

## trainer

In [10]:
batch_size = 16

s_training_args = DistillTrainingArguments(output_dir='distilbert-base-uncased-ft-clinc',
                                     eval_strategy='epoch',
                                     num_train_epochs=5,
                                     learning_rate=1e-5,
                                     per_device_train_batch_size=batch_size,
                                     per_device_eval_batch_size=batch_size, 
                                     alpha=0.5, 
                                     weight_decay=0.01,
                                     logging_strategy='epoch',
                                     report_to='none')



“”“
 一个关键的点在于必须显示传入 num_labels. 因为模型默认为二分类问题,而我们处理的是多分类问题, 所以在计算loss时会导致错误

/pytorch/aten/src/ATen/native/cuda/Loss.cu:250: nll_loss_forward_reduce_cuda_kernel_2d: block: [0,0,0], thread: [5,0,0]
Assertion `t >= 0 && t < n_classes` failed.
”“”

s_config = AutoConfig.from_pretrained(s_ckpt, num_labels=num_labels, 
                                      id2label=t_model.config.id2label, label2id=t_model.config.label2id)

In [11]:
s_model = AutoModelForSequenceClassification.from_pretrained(s_ckpt, config=s_config).to(device)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
import evaluate
accuracy_score = evaluate.load('accuracy')

def compute_metrics(pred):
    predictions, labels = pred
    predictions = np.argmax(predictions, axis=-1)
    return accuracy_score.compute(references=labels, predictions=predictions)

In [13]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=s_tokenizer)


In [14]:
distill_trainer = DistillTrainer(model=s_model,
                                 teacher_model=t_model, 
                                 args=s_training_args,
                                 data_collator=data_collator,
                                 train_dataset=clinc_enc['train'], 
                                 eval_dataset=clinc_enc['validation'], 
                                 compute_metrics=compute_metrics, 
                                 processing_class=s_tokenizer)
distill_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,2.612100,1.989118,0.690645
2,1.637400,1.197984,0.820968
3,1.039000,0.797102,0.885484
4,0.750200,0.636215,0.911290
5,0.633400,0.590316,0.917097


TrainOutput(global_step=4770, training_loss=1.3344299444362553, metrics={'train_runtime': 253.7998, 'train_samples_per_second': 300.434, 'train_steps_per_second': 18.794, 'total_flos': 362436621606108.0, 'train_loss': 1.3344299444362553, 'epoch': 5.0})

> ## 验证

In [17]:
predictions = distill_trainer.predict(clinc_enc['test'])
predictions

PredictionOutput(predictions=array([[-1.4919848 , -1.7534773 , -2.3053744 , ..., -1.5591881 ,
        -0.87042475, -1.487926  ],
       [-1.3114967 , -0.8170255 , -2.5876298 , ..., -1.4065212 ,
        -1.081182  , -1.7672105 ],
       [-0.5767546 ,  0.7511116 , -3.090537  , ..., -2.3759959 ,
        -0.98145646, -2.3256316 ],
       ...,
       [-2.2646203 , -2.6274161 , -2.176762  , ..., -0.5388781 ,
        -1.2604572 , -2.060916  ],
       [-2.5443778 , -2.7727022 , -2.1588743 , ..., -2.4869683 ,
        -2.2651107 , -1.6569507 ],
       [-2.5328217 , -2.3200157 , -2.2552571 , ..., -1.792756  ,
        -1.414095  , -1.8823116 ]], dtype=float32), label_ids=array([61, 61, 61, ..., 42, 42, 42]), metrics={'test_loss': 0.7436686158180237, 'test_accuracy': 0.8396363636363636, 'test_runtime': 6.8232, 'test_samples_per_second': 806.073, 'test_steps_per_second': 50.416})

In [18]:
compute_metrics((predictions.predictions,predictions.label_ids))['accuracy']

0.8396363636363636